**Bellman Optimal Policy**

In this exercise, we are going to find the Optimal Policy, using the Bellman equations, for the next MDP:

![alt text](two_state_mdp.png "Title")

Let us start with the imports. We use numpy and scipy.

In [1]:
import numpy as np
from scipy.optimize import fsolve

Now, let us define the parameters of the problem. We have an MDP with two states, and two actions. The rewards, discount factor, and transition probabilities are given in the figure above. We also define the four possible deterministic policies as $\pi_1$, $\pi_2$, $\pi_3$, and $\pi_4$:

In [2]:
gamma = 0.9
R = np.array([[-1, 0.6, 0.5, -0.9]]).T
P = np.array([[0.8, 0.2], [0.2, 0.8], [0.3, 0.7], [0.9, 0.1]])
pi_1 = np.array([[1, 0, 0, 0], [0, 0, 1, 0]])
pi_2 = np.array([[0, 1, 0, 0], [0, 0, 1, 0]])
pi_3 = np.array([[1, 0, 0, 0], [0, 0, 0, 1]])
pi_4 = np.array([[0, 1, 0, 0], [0, 0, 0, 1]])

Now, let us implement the Bellman equations, using the fixed point equations seen in the slides.
* $v^{\pi} = \left( I - \gamma \mathcal{P}^{\pi} \right)^{-1} \mathcal{R}^{\pi}$
* $q^{\pi} = \left( I - \gamma \mathcal{P} \Pi \right)^{-1} \mathcal{R}$
Note that, attending to the results, we know that $\pi_2$ is the optimal policy, as it provides the highest value, as we already know.

In [3]:
def bellman_equations(R, P, gamma, pi):
    # Code to be filled by the student
    num_states = pi.shape[0]
    I = np.eye(num_states)
    R_pi = np.dot(pi, R)
    P_pi = np.dot(pi, P)
    
    # Solve for v_pi
    v_pi = np.dot(np.linalg.inv(I - gamma * P_pi), R_pi)
    # Solve for q_pi
    q_pi = R + gamma * np.dot(P, v_pi)
    
    return v_pi, q_pi

for pi in [pi_1, pi_2, pi_3, pi_4]:
    v_pi, q_pi = bellman_equations(R, P, gamma, pi)
    with np.printoptions(precision=2, suppress=True):
        print(f"Policy = {pi.flatten()}")
        print(f"v^pi = {v_pi.flatten()}")
        print(f"q^pi = {q_pi.flatten()}")
r_opt = bellman_equations(R, P, gamma, pi_2)  # Save the optimal value for later
v_opt = r_opt[0].flatten()
q_opt = r_opt[1].flatten()

Policy = [1 0 0 0 0 0 1 0]
v^pi = [-5.09 -2.36]
q^pi = [-5.09 -2.02 -2.36 -5.24]
Policy = [0 1 0 0 0 0 1 0]
v^pi = [5.34 5.25]
q^pi = [3.79 5.34 5.25 3.9 ]
Policy = [1 0 0 0 0 0 0 1]
v^pi = [-9.83 -9.74]
q^pi = [-9.83 -8.19 -8.29 -9.74]
Policy = [0 1 0 0 0 0 0 1]
v^pi = [-0.63 -1.55]
q^pi = [-1.73 -0.63 -0.64 -1.55]


Now, we are going to obtain the optimal value function using the fsolve method from scipy.optimize. This method finds the root of a function, which is exactly what we need.

In [4]:
def bellman_opt_equations(v):
    v = v.reshape((2, 1))
    Ru = R[0:2]
    Rm = R[2:4]
    Pu = P[0:2, :]
    Pm = P[2:4, :]
    aux_term = np.array([np.amax(Ru + gamma * Pu @ v), np.amax(Rm + gamma * Pm @ v)])
    return v.flatten() - aux_term
v = fsolve(bellman_opt_equations, np.zeros((2, 1)))
with np.printoptions(precision=2, suppress=True):
    print(f"Optimal value found v* = {v.flatten()}, optimal value v_opt = {v_opt.flatten()}")

Optimal value found v* = [5.34 5.25], optimal value v_opt = [5.34 5.25]


Now, you have to do the same with the state-action value function.

In [5]:
def bellman_opt_equations_q(q):
    # Code to be filled by the student
    # q comes in as a flat array of size 4: [q(s0, a0), q(s0, a1), q(s1, a0), q(s1, a1)]
    
    # v*(s) is the maximum of q*(s, a) over all actions a
    v_s0 = np.maximum(q[0], q[1])
    v_s1 = np.maximum(q[2], q[3])
    
    # Format v into a column vector (2x1) for matrix multiplication
    v = np.array([[v_s0], [v_s1]])
    
    # Calculate the Bellman optimality update: R + gamma * P * v
    q_update = R + gamma * np.dot(P, v)
    
    # fsolve finds roots, so we return the residual (q - target)
    return q - q_update.flatten()

# Note: Renamed the function slightly to avoid overwriting the 'v' version, 
# updated fsolve to call 'bellman_opt_equations_q'
q_star = fsolve(bellman_opt_equations_q, np.zeros((4)))

with np.printoptions(precision=2, suppress=True):
    print(f"Optimal value found q* = {q_star.flatten()}, optimal value q_opt = {q_opt.flatten()}")

Optimal value found q* = [3.79 5.34 5.25 3.9 ], optimal value q_opt = [3.79 5.34 5.25 3.9 ]
